# Session 09: Lab Practice

### Libraries

In [1]:
# Libraries
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

### Upload data

In [2]:
# Upload CSV
train = pd.read_csv("sent_train.csv")
valid = pd.read_csv("sent_valid.csv")

### Text Preprocessing

In [3]:
# From capital to small letters
train["text"] = train["text"].astype(str).str.lower()
valid["text"] = valid["text"].astype(str).str.lower()

In [4]:
# Encode labels
encoder = LabelEncoder()
train["label"] = encoder.fit_transform(train["label"])
valid["label"] = encoder.transform(valid["label"])

### Tokenization

In [5]:
tokenizer = Tokenizer(num_words = 10000) 
tokenizer.fit_on_texts(train["text"])

# Text -> sequence 
X_train = tokenizer.texts_to_sequences(train["text"])
X_valid = tokenizer.texts_to_sequences(valid["text"])

# Padding
max_len = 40
X_train = pad_sequences(X_train, maxlen = max_len)
X_valid = pad_sequences(X_valid, maxlen = max_len)

# One-hot encoding
y_train = to_categorical(train["label"], num_classes = 3)
y_valid = to_categorical(valid["label"], num_classes = 3)

### LSTM Model

In [9]:
model = Sequential()
model.add(
    Embedding(
        input_dim = 10000,
        output_dim = 128,
    )
)

model.add(LSTM(64))
model.add(Dense(3, activation="softmax"))

model.compile(
    loss = "categorical_crossentropy",
    optimizer = "adam",
    metrics = ["accuracy"]
)
model.build(input_shape = (None, max_len))

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 40, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,603 (5.07 MB)

 Trainable params: 1,329,603 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

### Train

In [10]:
historty = model.fit(
    X_train, 
    y_train,
    validation_data = (X_valid, y_valid),
    epochs = 5,
    batch_size = 32
)

Epoch 1/5
299/299 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.7032 - loss: 0.7152 - val_accuracy: 0.8003 - val_loss: 0.5475
Epoch 2/5
299/299 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8522 - loss: 0.3885 - val_accuracy: 0.8074 - val_loss: 0.5084
Epoch 3/5
299/299 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.9265 - loss: 0.2121 - val_accuracy: 0.8116 - val_loss: 0.5655
Epoch 4/5
299/299 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.9590 - loss: 0.1264 - val_accuracy: 0.7969 - val_loss: 0.6617
Epoch 5/5
299/299 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9766 - loss: 0.0726 - val_accuracy: 0.8007 - val_loss: 0.7703


## Validation

In [ ]:
pred = model.predict(X_valid).argmax(axis = 1)
true = valid["label"].values

print(classification_report(true, pred, target_names = encoder.classes_))